[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_02/MFM_ch2_predictability/MFM_ch2_predictability.ipynb)

# MFM_ch2_predictability

Beyond the random walk: predictive regression of S&P 500 monthly excess returns on the log dividend-price ratio with the Stambaugh (1999) bias and a size simulation; out-of-sample R2 against the historical mean with the Clark-West test and Campbell-Thompson restrictions; local Whittle estimates of d = H - 0.5; the Escanciano-Lobato automatic portmanteau test; the December 2018 bank-tax event study for TLV and BRD with prediction-error variance and a portfolio test; Romano-Wolf StepM on 15 calendar tests; and a regression test of the Adaptive Market Hypothesis on rolling z*(5).

*Modelling Financial Markets (MFM), Chapter 2: Market Efficiency. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_2'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import io
import os
import re
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from scipy.special import gammaln
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Chart colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
GROUP_COL = {'Developed': MainBlue, 'Emerging/frontier': IDAred, 'Crypto': Amber, 'FX': Forest}

SEED = 42
TABLE_DIR = '.'


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'

# name -> (symbol, label, group, start date, days per year)
MARKETS = {
    'sp500':  ('GSPC.INDX',     'S&P 500 (US)',          'Developed', '2000-01-01', 252),
    'stoxx':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Developed', '2000-01-01', 252),
    'nikkei': ('N225.INDX',     'Nikkei 225 (Japan)',    'Developed', '2000-01-01', 252),
    'hsi':    ('HSI.INDX',      'Hang Seng (HK)',        'Developed', '2000-01-01', 252),
    'bet':    ('BET',           'BET (Romania)',         'Emerging/frontier', '2000-01-01', 252),
    'wig20':  ('WIG20.INDX',    'WIG20 (Poland)',        'Emerging/frontier', '2000-01-01', 252),
    'bux':    ('BUX.INDX',      'BUX (Hungary)',         'Emerging/frontier', '2000-01-01', 252),
    'px':     ('PX.INDX',       'PX (Czechia)',          'Emerging/frontier', '2000-01-01', 252),
    'xu100':  ('XU100.INDX',    'BIST 100 (Turkey)',     'Emerging/frontier', '2000-01-01', 252),
    'bvsp':   ('BVSP.INDX',     'Bovespa (Brazil)',      'Emerging/frontier', '2000-01-01', 252),
    'mxx':    ('MXX.INDX',      'IPC (Mexico)',          'Emerging/frontier', '2000-01-01', 252),
    'nsei':   ('NSEI.INDX',     'Nifty 50 (India)',      'Emerging/frontier', '2000-01-01', 252),
    'ssec':   ('SSEC.INDX',     'Shanghai (China)',      'Emerging/frontier', '2000-01-01', 252),
    'btc':    ('BTC-USD.CC',    'Bitcoin',               'Crypto', '2014-09-17', 365),
    'eth':    ('ETH-USD.CC',    'Ethereum',              'Crypto', '2016-01-01', 365),
    'eurron': ('REF:EUR',       'EUR/RON (reference rate)', 'FX', '2005-07-01', 252),
}

LABELS = {k: v[1] for k, v in MARKETS.items()}
GROUPS = {k: v[2] for k, v in MARKETS.items()}
PERIODS = {k: v[4] for k, v in MARKETS.items()}

_CACHE = {}


def read_market(symbol):
    """Daily price table of one series (local copy of the course data, otherwise the GitHub repository)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official RON reference rate (BNR annual XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END):
    """Daily closing price, cleaned with the chapter conventions."""
    symbol, _, group, start0, _ = MARKETS[name]
    start = start or start0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    s = read_market(symbol)['close'].loc[start:end]
    s = s[s > 0].dropna()
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    return s.rename(name)


def complete_months(x):
    """Drop the incomplete final month (last observation before the last business day of the month):
    the monthly analyses and the turn-of-the-month effect use complete months only."""
    last = x.index[-1]
    if last + pd.offsets.BMonthEnd(0) != last:
        x = x[x.index < last.to_period('M').start_time]
    return x


def log_returns(name, start=None, end=END):
    """Log returns on the series' own calendar."""
    return np.log(load_close(name, start, end)).diff().dropna().rename(name)


rets = {k: log_returns(k) for k in MARKETS}
ORDER = list(MARKETS)
for k in ORDER:
    print(f'{LABELS[k]:26s}', rets[k].index[0].date(), '->', rets[k].index[-1].date(), len(rets[k]), 'obs')

## Efficiency tests

In [ ]:
def variance_ratio(r, q):
    """Overlapping VR(q) with the Lo-MacKinlay (1988) bias corrections.

    Returns (VR, homoskedastic z, heteroskedasticity-robust z*).
    """
    x = np.asarray(r, dtype=float)
    T = len(x)
    mu = x.mean()
    e = x - mu
    var_a = (e ** 2).sum() / (T - 1)
    agg = np.convolve(x, np.ones(q), mode='valid') - q * mu        # overlapping q-period sums
    m = q * (T - q + 1) * (1 - q / T)
    var_c = (agg ** 2).sum() / m
    vr = var_c / var_a
    z_homo = (vr - 1) / np.sqrt(2 * (2 * q - 1) * (q - 1) / (3 * q * T))
    den = (e ** 2).sum() ** 2
    theta = 0.0
    for j in range(1, q):
        delta = T * (e[j:] ** 2 * e[:-j] ** 2).sum() / den
        theta += (2 * (q - j) / q) ** 2 * delta
    z_star = np.sqrt(T) * (vr - 1) / np.sqrt(theta)
    return vr, z_homo, z_star


def chow_denning(r, qs=(2, 5, 10, 20)):
    """Chow-Denning multiple test: max |z*(q)|; p-value from the studentized maximum modulus (infinite df)."""
    zs = [variance_ratio(r, q)[2] for q in qs]
    zmax = np.max(np.abs(zs))
    p = 1 - (2 * stats.norm.cdf(zmax) - 1) ** len(qs)
    return zmax, p, zs


def runs_test(r):
    """Number of sign runs (zeros dropped), with its mean and variance under independence."""
    s = np.sign(np.asarray(r, dtype=float))
    s = s[s != 0]
    n1, n2 = (s > 0).sum(), (s < 0).sum()
    n = n1 + n2
    runs = 1 + (s[1:] != s[:-1]).sum()
    mean = 2 * n1 * n2 / n + 1
    var = 2 * n1 * n2 * (2 * n1 * n2 - n) / (n ** 2 * (n - 1))
    z = (runs - mean) / np.sqrt(var)
    return runs, mean, z, 2 * (1 - stats.norm.cdf(abs(z)))


def robust_ljung_box(r, m=10):
    """Ljung-Box Q(m) and Q~(m) with the robust variance tau_k = sum e_t^2 e_{t-k}^2 / (sum e_t^2)^2."""
    e = np.asarray(r, dtype=float) - np.mean(r)
    T = len(e)
    s2 = (e ** 2).sum()
    q_lb, q_rob = 0.0, 0.0
    for k in range(1, m + 1):
        rho = (e[k:] * e[:-k]).sum() / s2
        tau = (e[k:] ** 2 * e[:-k] ** 2).sum() / s2 ** 2
        q_lb += T * (T + 2) * rho ** 2 / (T - k)
        q_rob += rho ** 2 / tau
    return q_lb, 1 - stats.chi2.cdf(q_lb, m), q_rob, 1 - stats.chi2.cdf(q_rob, m)


def _rs(x):
    y = np.cumsum(x - x.mean())
    s = x.std(ddof=0)
    return (y.max() - y.min()) / s if s > 0 else np.nan


def expected_rs(n):
    """E[R/S] for i.i.d. noise (Anis & Lloyd, 1976)."""
    if n <= 340:
        f = np.exp(gammaln((n - 1) / 2) - gammaln(n / 2)) / np.sqrt(np.pi)
    else:
        f = 1 / np.sqrt(n * np.pi / 2)
    i = np.arange(1, n)
    return f * np.sum(np.sqrt((n - i) / i))


def rs_hurst(r, min_n=10, n_sizes=20):
    """H from the slope of log(R/S) on log(n) (Hurst, 1951) and the Anis-Lloyd corrected H (slope of R/S - E[R/S], plus 0.5)."""
    x = np.asarray(r, dtype=float)
    T = len(x)
    sizes = np.unique(np.logspace(np.log10(min_n), np.log10(T // 4), n_sizes).astype(int))
    rs, ers = [], []
    for n in sizes:
        k = T // n
        vals = [_rs(x[i * n:(i + 1) * n]) for i in range(k)]
        rs.append(np.nanmean(vals))
        ers.append(expected_rs(n))
    ls, lrs, lers = np.log(sizes), np.log(rs), np.log(ers)
    h = np.polyfit(ls, lrs, 1)[0]
    h_al = 0.5 + np.polyfit(ls, lrs - lers, 1)[0]
    return h, h_al, pd.DataFrame({'n': sizes, 'rs': rs, 'expected_rs': ers})


def lo_modified_rs(r, q=None):
    """Lo's (1991) statistic V = Q_T / sqrt(T); under H0 (no long memory) V lies in [0.809, 1.862] with 95% probability."""
    x = np.asarray(r, dtype=float)
    T = len(x)
    e = x - x.mean()
    if q is None:                                   # Andrews (1991) bandwidth for an AR(1)
        rho = np.corrcoef(e[1:], e[:-1])[0, 1]
        q = int(np.floor((3 * T / 2) ** (1 / 3) * abs(2 * rho / (1 - rho ** 2)) ** (2 / 3)))
    s2 = (e ** 2).mean()
    for j in range(1, q + 1):
        w = 1 - j / (q + 1)
        s2 += 2 * w * (e[j:] * e[:-j]).sum() / T
    y = np.cumsum(e)
    Q = (y.max() - y.min()) / np.sqrt(s2)
    return Q / np.sqrt(T), q


def dfa_hurst(r, min_box=10, n_sizes=18, max_frac=0.25):
    """DFA-1 exponent: slope of log F(n) on log n for the cumulative profile of demeaned returns."""
    x = np.asarray(r, dtype=float)
    y = np.cumsum(x - x.mean())
    T = len(y)
    sizes = np.unique(np.logspace(np.log10(min_box), np.log10(int(T * max_frac)), n_sizes).astype(int))
    F = []
    for n in sizes:
        k = T // n
        seg = y[:k * n].reshape(k, n)
        t = np.arange(n)
        A = np.vstack([t, np.ones(n)]).T
        coef, *_ = np.linalg.lstsq(A, seg.T, rcond=None)
        resid = seg.T - A @ coef
        F.append(np.sqrt((resid ** 2).mean()))
    return np.polyfit(np.log(sizes), np.log(F), 1)[0], pd.DataFrame({'n': sizes, 'F': F})


def rolling_stat(r, func, window=500, step=21):
    """Apply func on rolling windows of `window` observations, every `step` observations."""
    x = r.values
    idx, out = [], []
    for end in range(window, len(x) + 1, step):
        idx.append(r.index[end - 1])
        out.append(func(x[end - window:end]))
    return pd.Series(out, index=idx)

## Helper functions

In [ ]:
SHILLER_URLS = ['http://www.econ.yale.edu/~shiller/data/ie_data.xls']

FRED = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id='

CAL_MARKETS = ['sp500', 'bet', 'stoxx', 'wig20', 'bux']

_R = np.column_stack([np.ones(4), -np.eye(4)])

def _get(url, timeout=120, agent='Mozilla/5.0'):
    return urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': agent}),
                                  timeout=timeout).read()


def shiller():
    """Monthly S&P Composite (Robert J. Shiller's public data): P = monthly average price,
    D = trailing 12-month dividends (nominal)."""
    if 'shiller' in _CACHE:
        return _CACHE['shiller']
    urls = []
    try:
        html = _get('https://shillerdata.com/', 60).decode('utf-8', 'ignore')
        urls = ['https:' + u if u.startswith('//') else u for u in re.findall(r'href="([^"]*ie_data\.xls[^"]*)"', html)]
    except Exception:
        pass
    raw = None
    for url in urls + SHILLER_URLS:
        try:
            raw = _get(url)
            break
        except Exception:
            continue
    d = pd.read_excel(io.BytesIO(raw), sheet_name='Data', header=None, skiprows=8).iloc[:, [0, 1, 2]]
    d.columns = ['date', 'P', 'D']
    d = d[pd.to_numeric(d['date'], errors='coerce').notna()].copy()
    yr = d['date'].astype(float)
    year = np.floor(yr + 1e-9).astype(int)
    month = np.round((yr - year) * 100).astype(int)
    d.index = pd.to_datetime(dict(year=year, month=month, day=1)) + pd.offsets.MonthEnd(0)
    d = d[['P', 'D']].apply(pd.to_numeric, errors='coerce').dropna()
    _CACHE['shiller'] = d
    return d


def fred(series):
    """FRED series (no API key), as a pd.Series indexed at month ends."""
    d = pd.read_csv(io.BytesIO(_get(FRED + series, agent='Python-urllib')), index_col=0, parse_dates=True)
    s = pd.to_numeric(d.iloc[:, 0], errors='coerce').dropna()
    s.index = s.index + pd.offsets.MonthEnd(0)
    return s.rename(series)


def sp500_monthly(end='2026-08-31'):
    """Monthly S&P 500, P (monthly average) and D (trailing 12-month dividends), extended to `end`.
    Splicing: P = monthly average of the daily S&P 500 closes; D = trailing 12-month dividends of the SPY fund
    (from the adjusted and unadjusted closes), scaled by the mean ratio D_Shiller / D_SPY over the last 12 common months."""
    sh = shiller()
    spx = read_market('GSPC.INDX')['close']
    P_ext = spx.resample('ME').mean()
    spy = read_market('SPY.US')
    tr = spy['adjusted_close'] / spy['adjusted_close'].shift(1)
    div = (tr * spy['close'].shift(1) - spy['close']).clip(lower=0)
    div[div < 1e-3 * spy['close']] = 0.0                       # ex-dividend days only
    D_spy = div.resample('ME').sum().rolling(12).sum()
    common = sh.index.intersection(D_spy.dropna().index)[-12:]
    kP = float((sh.loc[common, 'P'] / P_ext.loc[common]).mean())
    kD = float((sh.loc[common, 'D'] / D_spy.loc[common]).mean())
    last = sh.index[-1]
    ext = pd.DataFrame({'P': kP * P_ext, 'D': kD * D_spy}).loc[last + pd.offsets.MonthEnd(1):end]
    out = pd.concat([sh, ext]).loc[:end]
    out.attrs.update(shiller_last=str(last.date()), kP=kP, kD=kD)
    return out

## Analysis

In [ ]:
def monthly_data(start='1934-01-31', end='2026-08-31'):
    """S&P 500 monthly excess log return (with dividends) and log D/P at the end of the previous month."""
    d = sp500_monthly(end)
    rf = fred('TB3MS') / 100                                   # annualised 3-month T-bill rate
    r = np.log((d['P'] + d['D'] / 12) / d['P'].shift(1))
    rf_m = np.log(1 + rf / 12).reindex(d.index).shift(1)       # known at the start of the month
    out = pd.DataFrame({'ex': r - rf_m, 'dp': np.log(d['D'] / d['P'])})
    out['dp_lag'] = out['dp'].shift(1)
    return out.loc[start:end].dropna(), d.attrs


def predictive_regression(df, nw_lags=12):
    """OLS with Newey-West errors, AR(1) regression of the predictor, innovation correlation and the Stambaugh correction."""
    y, x = df['ex'].values, df['dp_lag'].values
    X = sm.add_constant(x)
    ols = sm.OLS(y, X).fit()
    hac = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})
    ar = sm.OLS(df['dp'].values, X).fit()                      # dp_t on dp_{t-1}
    u, v = ols.resid, ar.resid
    T = len(y)
    rho = ar.params[1]
    s_uv, s_vv = np.cov(u, v)[0, 1], np.var(v, ddof=1)
    bias_rho = -(1 + 3 * rho) / T                               # Kendall (1954)
    bias_beta = s_uv / s_vv * bias_rho                          # Stambaugh (1999)
    return dict(T=T, start=str(df.index[0].date()), end=str(df.index[-1].date()),
                beta=ols.params[1], se_ols=ols.bse[1], t_ols=ols.tvalues[1], se_nw=hac.bse[1], t_nw=hac.tvalues[1],
                r2=ols.rsquared, rho=rho, corr_uv=np.corrcoef(u, v)[0, 1], ratio_uv=s_uv / s_vv,
                bias_rho=bias_rho, bias_beta=bias_beta, beta_adj=ols.params[1] - bias_beta,
                t_adj=(ols.params[1] - bias_beta) / hac.bse[1])


def simulate_stambaugh(rho, corr_uv, T, n_sim=2000, seed=SEED):
    """Bias of beta and size of the 5% t-test when beta = 0 (persistent predictor, correlated innovations)."""
    rng = np.random.default_rng(seed)
    betas, rej = [], 0
    for _ in range(n_sim):
        e = rng.multivariate_normal([0, 0], [[1, corr_uv], [corr_uv, 1]], T + 1)
        x = np.zeros(T + 1)
        for t in range(1, T + 1):
            x[t] = rho * x[t - 1] + e[t, 1]
        y = e[1:, 0]                                             # beta = 0
        f = sm.OLS(y, sm.add_constant(x[:-1])).fit()
        betas.append(f.params[1] / 1.0)
        rej += abs(f.tvalues[1]) > 1.96
    return dict(mean_beta=float(np.mean(betas)), size=rej / n_sim)


def oos_r2(df, oos_start='1970-01-31', nw_lags=12):
    """Recursive (expanding-window) forecasts: predictive regression vs historical mean.
    R2_OS (Welch-Goyal), Clark-West, and the version with Campbell-Thompson restrictions."""
    rows = []
    idx = df.index
    first = idx.get_loc(idx[idx >= oos_start][0])
    for i in range(first, len(df)):
        est = df.iloc[:i]
        b = np.polyfit(est['dp_lag'], est['ex'], 1)
        f_pred = b[1] + b[0] * df['dp_lag'].iloc[i]
        f_ct = max(b[1] + max(b[0], 0.0) * df['dp_lag'].iloc[i], 0.0) if b[0] > 0 else max(est['ex'].mean(), 0.0)
        rows.append((idx[i], df['ex'].iloc[i], est['ex'].mean(), f_pred, f_ct))
    o = pd.DataFrame(rows, columns=['date', 'y', 'mean', 'pred', 'ct']).set_index('date')
    e0, e1, e2 = o['y'] - o['mean'], o['y'] - o['pred'], o['y'] - o['ct']
    r2 = 1 - (e1 ** 2).sum() / (e0 ** 2).sum()
    r2_ct = 1 - (e2 ** 2).sum() / (e0 ** 2).sum()
    f = e0 ** 2 - (e1 ** 2 - (o['mean'] - o['pred']) ** 2)            # Clark-West
    cw = sm.OLS(f.values, np.ones(len(f))).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})
    dm = sm.OLS((e0 ** 2 - e1 ** 2).values, np.ones(len(f))).fit(cov_type='HAC', cov_kwds={'maxlags': nw_lags})
    o['cum_dsse'] = (e0 ** 2 - e1 ** 2).cumsum()
    o['cum_dsse_ct'] = (e0 ** 2 - e2 ** 2).cumsum()
    sr_m = df['ex'].mean() / df['ex'].std()                       # unconditional monthly Sharpe ratio
    binds = float(((o['ct'] - o['pred']).abs() > 1e-12).mean())       # how often the restrictions bind
    ct_gain = lambda r2_: r2_ / (1 - r2_) * (1 + sr_m ** 2) / sr_m ** 2
    return o, dict(n=len(o), start=str(o.index[0].date()), r2=r2, r2_ct=r2_ct, cw_t=cw.tvalues[0],
                   cw_p=1 - stats.norm.cdf(cw.tvalues[0]), dm_t=dm.tvalues[0], sr_m=sr_m,
                   gain_05=ct_gain(0.005), ct_binds=binds, gain_ct=ct_gain(r2_ct) if r2_ct > 0 else float('nan'))


def fig_oos(o):
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    ax.plot(o.index, o['cum_dsse'], color=IDAred, lw=1.0, label='Predictive regression on log D/P vs historical mean')
    ax.axhline(0, color=Gray, lw=0.7, ls='--')
    ax.set_ylabel('Cumulative reduction in squared\nforecast errors vs historical mean')
    ax.set_title('Out-of-sample: does the dividend-price ratio beat the historical mean? (S&P 500, monthly)',
                 fontsize=9, loc='left')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.13), ncol=1, frameon=False)
    plt.tight_layout()
    save_fig('ch2_oos_dp')


def local_whittle(x, m):
    """d estimated by minimising the local Whittle function (Robinson, 1995); asymptotic SE 1/(2 sqrt(m))."""
    x = np.asarray(x, dtype=float)
    T = len(x)
    w = np.fft.fft(x - x.mean())
    lam = 2 * np.pi * np.arange(1, m + 1) / T
    I = np.abs(w[1:m + 1]) ** 2 / (2 * np.pi * T)
    from scipy.optimize import minimize_scalar
    R = lambda d: np.log(np.mean(lam ** (2 * d) * I)) - 2 * d * np.mean(np.log(lam))
    d = minimize_scalar(R, bounds=(-0.49, 0.99), method='bounded').x
    return d, 1 / (2 * np.sqrt(m))


def el_autoportmanteau(x, d_max=10, q=2.4):
    """AQ = T sum_{j<=p~} rho~_j^2 with rho~_j^2 = gamma_j^2 / tau_j (robust); p~ chosen from the data; AQ ~ chi2(1)."""
    y = np.asarray(x, dtype=float)
    T = len(y)
    e = y - y.mean()
    g0 = (e ** 2).mean()
    rt = []
    rho_raw = []
    for j in range(1, d_max + 1):
        gj = (e[j:] * e[:-j]).sum() / (T - j)
        tj = (e[j:] ** 2 * e[:-j] ** 2).sum() / (T - j)
        rt.append(gj ** 2 / tj)
        rho_raw.append(gj / g0)
    rt = np.array(rt)
    Q = T * np.cumsum(rt)
    p = np.arange(1, d_max + 1)
    pen = p * np.log(T) if np.sqrt(T * np.max(rt)) <= np.sqrt(q * np.log(T)) else 2 * p   # switch based on the robust correlations
    L = Q - pen
    pt = int(p[np.argmax(L)])
    AQ = Q[pt - 1]
    return AQ, pt, 1 - stats.chi2.cdf(AQ, 1)


def event_study_bank_tax(event='2018-12-19', est=(-250, -11), win=(0, 5)):
    """Market model on [-250, -11]; abnormal returns with the prediction-error variance; portfolio test vs independence."""
    px = pd.concat({'TLV': read_market('TLV.RO')['adjusted_close'], 'BRD': read_market('BRD.RO')['adjusted_close'],
                    'BET': read_market('BET')['close']}, axis=1).dropna()        # align prices on common trading days
    px = px[px.index.dayofweek < 5]
    r = np.log(px).diff().dropna()
    t0 = r.index.get_loc(pd.Timestamp(event))
    E = r.iloc[t0 + est[0]:t0 + est[1] + 1]
    W = r.iloc[t0 + win[0]:t0 + win[1] + 1]
    rm, mu_m, var_m, T0 = W['BET'], E['BET'].mean(), E['BET'].var(ddof=1), len(E)
    out, resid = {}, {}
    for k in ['TLV', 'BRD', 'PORT']:
        yE = E[['TLV', 'BRD']].mean(axis=1) if k == 'PORT' else E[k]
        yW = W[['TLV', 'BRD']].mean(axis=1) if k == 'PORT' else W[k]
        f = sm.OLS(yE, sm.add_constant(E['BET'])).fit()
        s2 = f.mse_resid
        resid[k] = f.resid
        ar = yW - f.params['const'] - f.params['BET'] * rm
        pe = s2 * (1 + 1 / T0 + (rm - mu_m) ** 2 / ((T0 - 1) * var_m))          # prediction-error variance
        car = ar.sum()
        # CAR: sum of prediction variances + covariance induced by parameter estimation (MacKinlay, 1997)
        Xs = np.column_stack([np.ones(len(rm)), rm.values])
        XE = np.column_stack([np.ones(T0), E['BET'].values])
        V = s2 * (np.eye(len(rm)) + Xs @ np.linalg.inv(XE.T @ XE) @ Xs.T)
        out[k] = dict(beta=f.params['BET'], sigma=np.sqrt(s2), ar0=ar.iloc[0], t0_naive=ar.iloc[0] / np.sqrt(s2),
                      t0=ar.iloc[0] / np.sqrt(pe.iloc[0]), infl0=pe.iloc[0] / s2, car=car,
                      t_car=car / np.sqrt(V.sum()), t_car_naive=car / np.sqrt(s2 * len(rm)),
                      var_car=V.sum(), car_post=ar.iloc[1:].sum(), t_car_post=ar.iloc[1:].sum() / np.sqrt(V[1:, 1:].sum()))
    rbar = np.corrcoef(resid['TLV'], resid['BRD'])[0, 1]
    # joint test assuming independence: mean AR0 / (sqrt(s1^2 + s2^2)/2)
    # under independence: the same prediction-error variances as for the portfolio, no covariance between firms
    s_ind = np.sqrt(out['TLV']['sigma'] ** 2 * out['TLV']['infl0'] + out['BRD']['sigma'] ** 2 * out['BRD']['infl0']) / 2
    ar0_mean = (out['TLV']['ar0'] + out['BRD']['ar0']) / 2
    car_mean = (out['TLV']['car'] + out['BRD']['car']) / 2
    s_car_ind = np.sqrt(out['TLV']['var_car'] + out['BRD']['var_car']) / 2
    return dict(event=event, rm0=float(rm.iloc[0]), T0=T0, rbar=rbar, firms=out,
                ar0_mean=ar0_mean, t_indep=ar0_mean / s_ind, t_port=out['PORT']['t0'],
                car_mean=car_mean, tcar_indep=car_mean / s_car_ind, tcar_port=out['PORT']['t_car'],
                kp_factor=np.sqrt((1 - rbar) / (1 + rbar)),
                varmean10=1 + 9 * rbar,                       # Var(mean of 10 ARs) / Var under independence
                kp_inflation10=(1 + 9 * rbar) / (1 - rbar))  # overstatement of the variance of the cross-sectional statistic


def _nw_se(X, u, L=5):
    """Newey-West covariance matrix (Bartlett kernel, L lags) for OLS."""
    T = len(u)
    Xu = X * u[:, None]
    S = Xu.T @ Xu / T
    for j in range(1, L + 1):
        G = Xu[j:].T @ Xu[:-j] / T
        S += (1 - j / (L + 1)) * (G + G.T)
    XX = np.linalg.inv(X.T @ X / T)
    return XX @ S @ XX / T


def _cal_est(r, dow, month):
    """Estimates and HAC covariances of the 3 effects for one market.
    dow: weekday of each observation; month: integer month label, (block or year) * 100 + month."""
    D = np.column_stack([(dow == d).astype(float) for d in range(5)])
    grp = pd.Series(np.arange(len(r))).groupby(month)
    rank = grp.cumcount().values
    rank_end = grp.cumcount(ascending=False).values
    tom = ((rank < 3) | (rank_end == 0)).astype(float)
    bd = np.linalg.lstsq(D, r, rcond=None)[0]
    Vd = _nw_se(D, r - D @ bd, 5)
    X = np.column_stack([np.ones(len(r)), tom])
    bt = np.linalg.lstsq(X, r, rcond=None)[0]
    Vt = _nw_se(X, r - X @ bt, 5)
    ms = np.expm1(pd.Series(r).groupby(month).sum())          # simple monthly returns, as in january_regression
    jan = (np.asarray(ms.index) % 100 == 1).astype(float)
    Xm = np.column_stack([np.ones(len(ms)), jan])
    bj = np.linalg.lstsq(Xm, ms.values, rcond=None)[0]
    Vj = _nw_se(Xm, ms.values - Xm @ bj, 3)
    return dict(dow=(bd, Vd), tom=(bt[1], Vt[1, 1]), jan=(bj[1], Vj[1, 1]))


def _cal_p(est, center=None):
    """p-values: Wald (4 df) for the day of the week, two-sided z for January and TOM.
    center: the estimates on the original data (bootstrap statistic centred, as in Romano-Wolf)."""
    out = {}
    bd, Vd = est['dow']
    d = _R @ (bd - (center['dow'][0] if center else 0))
    out['dow'] = 1 - stats.chi2.cdf(float(d @ np.linalg.solve(_R @ Vd @ _R.T, d)), 4)
    for e in ['jan', 'tom']:
        b, v = est[e]
        z = (b - (center[e][0] if center else 0)) / np.sqrt(v)
        out[e] = 2 * (1 - stats.norm.cdf(abs(z)))
    return out


def stepm_calendar(n_boot=999, block=3, alpha=0.05, seed=SEED):
    """Romano-Wolf StepM with the min-p statistic: the SAME blocks of calendar months are resampled for
    all markets (keeps the dependence across markets and the calendar structure within the month);
    the bootstrap statistics are centred at the original estimates."""
    rets = {k: complete_months(log_returns(k)) for k in CAL_MARKETS}
    months = pd.period_range('2000-01', max(r.index[-1] for r in rets.values()).to_period('M'), freq='M')
    orig, p0, pos = {}, {}, {}
    for k, r in rets.items():
        mlab = r.index.to_period('M')
        orig[k] = _cal_est(r.values, r.index.dayofweek.values, r.index.year.values * 100 + r.index.month.values)
        for e, pv in _cal_p(orig[k]).items():
            p0[(k, e)] = pv
        pos[k] = {m: np.where(mlab == m)[0] for m in months}
    keys = [(k, e) for k in CAL_MARKETS for e in ['dow', 'jan', 'tom']]
    mnum = np.array([m.month for m in months])
    rng = np.random.default_rng(seed)
    nb = int(np.ceil(len(months) / block))
    boot_p = np.zeros((n_boot, len(keys)))
    for b in range(n_boot):
        starts = rng.integers(0, len(months) - block + 1, nb)
        msel = np.concatenate([np.arange(s0, s0 + block) for s0 in starts])[:len(months)]
        for k, r in rets.items():
            parts = [pos[k][months[m]] for m in msel]
            idx = np.concatenate(parts)
            lab = np.concatenate([np.full(len(pp), j) for j, pp in enumerate(parts)])
            newlab = lab * 100 + mnum[msel][lab]          # label = block position * 100 + original month
            est = _cal_est(r.values[idx], r.index.dayofweek.values[idx], newlab)
            for e, pv in _cal_p(est, orig[k]).items():
                boot_p[b, keys.index((k, e))] = pv
    p = np.array([p0[k] for k in keys])
    active = np.ones(len(keys), bool)
    rejected = np.zeros(len(keys), bool)
    steps = []
    while active.any():
        crit = np.quantile(boot_p[:, active].min(axis=1), alpha)
        new = active & (p <= crit)
        steps.append(float(crit))
        if not new.any():
            break
        rejected |= new
        active &= ~new
    holm = np.zeros(len(keys), bool)
    for i, ix in enumerate(np.argsort(p)):
        if p[ix] < alpha / (len(p) - i):
            holm[ix] = True
        else:
            break
    return pd.DataFrame({'market': [k for k, _ in keys], 'effect': [e for _, e in keys], 'p': p,
                         'holm_reject': holm, 'stepm_reject': rejected,
                         'boot_size5': (boot_p < 0.05).mean(axis=0)}), steps


def amh_regression(k, window=500, step=21):
    r = log_returns(k)
    z = rolling_stat(r, lambda x: variance_ratio(x, 5)[2], window, step)
    vol = rolling_stat(r, lambda x: np.std(x, ddof=1) * np.sqrt(252), window, step)
    starts = [r.index[r.index.get_loc(d) - window + 1] for d in z.index]
    crisis = [((s <= pd.Timestamp('2009-06-30')) & (e >= pd.Timestamp('2008-09-01'))) |
              ((s <= pd.Timestamp('2020-06-30')) & (e >= pd.Timestamp('2020-02-15'))) for s, e in zip(starts, z.index)]
    X = sm.add_constant(pd.DataFrame({'vol': 100 * vol.values, 'crisis': np.array(crisis, float)}))
    f = sm.OLS(z.values, X).fit(cov_type='HAC', cov_kwds={'maxlags': int(np.ceil(window / step))})
    return dict(n=len(z), b_vol=f.params['vol'], t_vol=f.tvalues['vol'], b_crisis=f.params['crisis'],
                t_crisis=f.tvalues['crisis'], r2=f.rsquared, lags=int(np.ceil(window / step)))

## Run

In [ ]:
df, attrs = monthly_data()
print(attrs)
pr = predictive_regression(df)
print(pr)
print(simulate_stambaugh(pr['rho'], pr['corr_uv'], 360))
o, res = oos_r2(df)
print(res)
fig_oos(o)
for k in ['sp500', 'bet', 'btc', 'eth']:
    x = rets[k].values
    print(k, local_whittle(x, int(len(x) ** 0.65)), el_autoportmanteau(x))
print(event_study_bank_tax())
print(amh_regression('sp500'))
print(amh_regression('bet'))
tab, steps = stepm_calendar()
print(tab.round(4))
print(steps)

![ch2_oos_dp](./ch2_oos_dp.png)

Output: `ch2_oos_dp.pdf`